Imports and dates

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add repo root to PYTHONPATH
HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]   # notebooks → repo root
sys.path.insert(0, str(PROJECT_ROOT))


from prm_opt.run_s25 import run_s25_s1, run_s25_s2, run_s25_s2_v2, run_s25_s2_v2_sensitivity
# from prm_opt.run_s26 import run_s26_s1, run_s26_s2
from prm_opt.outputs import build_run_report, print_run_report, build_run_report_s1, print_run_report_s1
from prm_opt.build_jobs import build_jobs
from prm_opt.ingest_s25 import ingest_s25
from prm_opt.config import PlanningToggles

# -----------------------------
# DATE RANGES
# -----------------------------
START_S25 = "2025-03-30"
END_S25   = "2025-10-26"

START_S25_DEBUG = "2025-06-01"
END_S25_DEBUG   = "2025-06-30"

START_S26 = "2025-03-29"
END_S26   = "2025-10-24"

Common toggles


In [ ]:

toggles = PlanningToggles(
    sla_buffer_mins=0,
    spill_bucket_cap=12,
    standby_dep_vert_mins=10,
    standby_arr_horiz_mins=10,
)


Run S25 Scenario 1 (baseline)

In [3]:

out_s1 = run_s25_s1(START_S25, END_S25, toggles=toggles)

report_s1 = build_run_report_s1(out_s1, day_from="s", hour_method="max")
print_run_report_s1(report_s1)

# # 15-min peak day detail:
# report_s1["peak_day_report"]["bucket_level"].to_csv("S1_peak_day_15min.csv")
# report_s1["peak_day_report"]["hourly"].to_csv("S1_peak_day_hourly.csv")





Unmatched passenger rows after merge (missing Chocks DT): 4437
Unique unmatched flight keys: 2699

Unmatched key reasons:
reason
no_flight_candidate               1935
scheduled_dt_mismatch              702
exact_match_should_have_joined      62
Name: count, dtype: int64

Dropped 4321 passenger rows due to reasons {'no_flight_candidate', 'scheduled_dt_mismatch'}

[BUILD_JOBS] Dropping 10 arrival jobs with expired SLA window (max_late_mins=180).
        Passenger ID Flight Number Airline Code      sla_start_time  \
18056       11148662          1460           BA 2025-05-02 00:01:00   
28566       11390224          6619           RK 2025-06-04 15:15:00   
54363       12013252           312           U2 2025-07-21 19:16:42   
70085       12405117          3252           EI 2025-08-06 11:12:05   
100710      13159410           748           LS 2025-09-30 01:35:15   
100711      13159411           748           LS 2025-09-30 01:35:15   
100714      13159414           748           LS 2025-

C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\prm_opt\outputs.py:756: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = bucket_level.resample("H").max()


Run S25 Scenario 2 (optimised)

In [ ]:

sens = run_s25_s2_v2_sensitivity(
    start=START_S25_DEBUG,
    end=END_S25_DEBUG,
    vertical_cycle_grid=[15, 0, 10],
    solver_name="highs",
    toggles=toggles,
    time_limit_sec=600,
    threads=8,
    mip_rel_gap=0.20,
)

# Build a comparison table

for r in sens["runs"]:
    print("\n" + "#" * 80)
    print(f"VERTICAL CYCLE SENSITIVITY: {r['vertical_cycle_mins']} mins")
    print("#" * 80)

    report = build_run_report(r)

    # attach assumption for printing (not rebuilding anything)
    report["assumptions"] = {
        "vertical_cycle_mins": r["vertical_cycle_mins"]
    }

    print_run_report(report)



PRM OPT — S25 Scenario 2 (v2) — Sensitivity
Window : 2025-06-01 → 2025-06-30
vertical_cycle_grid: [0, 5, 10, 15]

[1/4] ingest_s25…

Unmatched passenger rows after merge (missing Chocks DT): 638
Unique unmatched flight keys: 374

Unmatched key reasons:
reason
no_flight_candidate               324
scheduled_dt_mismatch              42
exact_match_should_have_joined      8
Name: count, dtype: int64

Dropped 618 passenger rows due to reasons {'scheduled_dt_mismatch', 'no_flight_candidate'}
    ✓ ingest_s25 done | rows=16,281   [24.28s]
[2/4] build_jobs…

[BUILD_JOBS] Dropping 1 arrival jobs with expired SLA window (max_late_mins=180).
     Passenger ID Flight Number Airline Code      sla_start_time  \
794      11390224          6619           RK 2025-06-04 15:15:00   

         Job Start Time  sla_limit Chocks DT  
794 2025-06-05 11:08:52         20       NaT  
    ✓ build_jobs done | jobs=16,280   [0.42s]
[3/4] build_tau + classes + spin_removed…
    ✓ params built | N_AMB=14   [0.61s]


MemoryError: bad allocation

In [ ]:

# out = run_s25_s2_v2(
#     start=START_S25_DEBUG,
#     end=END_S25_DEBUG,
#     solver_name="highs",
#     toggles=toggles,
#     solve_model=True,
#     time_limit_sec=600,
#     threads=8,
#     mip_rel_gap=0.20,  # optional: stop earlier with a usable solution
# )

# report = build_run_report(out)     # builds sanity + peak day hourly
# print_run_report(report)           # prints clean report


# # report["peak_day_report"]["hourly"].to_csv("peak_day_hourly_fleet.csv")

Run S26 Scenario 1

In [ ]:

# out_s26_s1 = run_s26_s1(
#     start=START_S26,
#     end=END_S26,
#     penetration_rates=penetration_rates,
#     ssr_mix=ssr_mix,
#     stand_actuals=stand_actuals,
#     stand_dist=stand_dist,
#     service_time_params=service_time_params,
#     chocks_offset_params=chocks_offset_params,
#     toggles=toggles,
# )

# out_s26_s1["summary"]
# out_s26_s1["ambulift_curve"].head()
# out_s26_s1["driver_curve"].head()


Run S26 scenario 2

In [ ]:

# out_s26_s2 = run_s26_s2(
#     start=START_S26,
#     end=END_S26,
#     penetration_rates=penetration_rates,
#     ssr_mix=ssr_mix,
#     stand_actuals=stand_actuals,
#     stand_dist=stand_dist,
#     service_time_params=service_time_params,
#     chocks_offset_params=chocks_offset_params,
#     solver_name="highs",
#     toggles=toggles,
# )

# out_s26_s2["summary"]
